In [ ]:
# load data here
#Enter answer here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import database
df= pd.read_csv('diabetes_dataset.csv')

y=df_final['diagnosed_diabetes']

X= df_final.drop(columns=['diagnosed_diabetes']).copy()

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42   # no stratify
)

In [ ]:
# Import required libraries
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

# --------------------------------------------------
# 1. Define the base estimator (weak learner)
#    AdaBoost typically works best with shallow trees
# --------------------------------------------------
base_estimator = DecisionTreeClassifier(random_state=42)

# --------------------------------------------------
# 2. Define the AdaBoost model
# --------------------------------------------------
ada = AdaBoostClassifier(
    estimator=base_estimator,   # weak learner
    random_state=42
)

# --------------------------------------------------
# 3. Define the hyperparameter grid
# --------------------------------------------------
param_grid = {
    # Number of boosting rounds
    "n_estimators": [50, 100, 200],

    # Contribution of each weak learner
    "learning_rate": [0.01, 0.1, 1.0],

    # Control complexity of the weak learner
    "estimator__max_depth": [1, 2, 3],
    "estimator__min_samples_leaf": [1, 5, 10],
}

# --------------------------------------------------
# 4. Cross-validation strategy
#    StratifiedKFold preserves class imbalance
# --------------------------------------------------
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# --------------------------------------------------
# 5. Define GridSearch
#    Choose scoring based on your goal:
#    - "recall"         → minimize false negatives
#    - "precision"      → minimize false positives
#    - "f1"             → balance precision & recall
#    - "balanced_accuracy" → good for imbalanced data
# --------------------------------------------------
grid_ada = GridSearchCV(
    estimator=ada,
    param_grid=param_grid,
    scoring="recall_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# --------------------------------------------------
# 6. Fit GridSearch to the data
# --------------------------------------------------
grid_ada.fit(X_train, y_train)

# --------------------------------------------------
# 7. Best model and results
# --------------------------------------------------
print("Best parameters found:")
print(grid_ada.best_params_)

print("\nBest CV score:")
print(grid_ada.best_score_)

best_ada = grid_ada.best_estimator_

# --------------------------------------------------
# 8. Evaluate on the full dataset (or test set)
# --------------------------------------------------
y_pred = best_ada.predict(X)

print("\nClassification report:")
print(classification_report(y, y_pred, digits=4))
